# Notebook 2: Transform data records into dataframes

Import the Python library dependencies.

In [8]:
import pathlib

import polars as pl
import watermark

Run a "watermark" to show which library versions are used in this notebook's runtime environment.

In [9]:
%load_ext watermark
%watermark
%watermark --iversions

Last updated: 2025-11-11T18:46:03.355621-08:00

Python implementation: CPython
Python version       : 3.13.8
IPython version      : 9.1.0

Compiler    : Clang 17.0.0 (clang-1700.0.13.3)
OS          : Darwin
Release     : 24.6.0
Machine     : arm64
Processor   : arm
CPU cores   : 14
Architecture: 64bit

watermark: 2.5.0
polars   : 1.29.0



## Parse the Open Sanctions dataset

In [2]:
df_src: pl.DataFrame = pl.read_ndjson("data/open-sanctions.json")

# normalize the IRIs for joining
df_src = df_src.replace_column(
    1,
    pl.Series(
        "id",
        map(lambda s: "sz:ds_open-sanctions_" + s, df_src["RECORD_ID"])
    ),
)

df_src = df_src.replace_column(
    2,
    pl.Series(
        "class",
        map(lambda s: "sz:" + s.capitalize(), df_src["RECORD_TYPE"])
    ),
)

In [3]:
df_src

DATA_SOURCE,id,class,LAST_CHANGE,NAMES,GENDER,RISKS,ADDRESSES,DATES,COUNTRIES,IDENTIFIERS,SOURCE_LINKS,RELATIONSHIPS,URL,CONTACTS
str,str,str,str,list[struct[3]],str,list[struct[1]],list[struct[7]],list[struct[2]],list[struct[3]],list[struct[9]],list[struct[1]],list[struct[5]],str,list[struct[1]]
"""OPEN-SANCTIONS""","""sz:ds_open-sanctions_NK-25vyVF…","""sz:Person""","""2024-07-30T16:41:14""","[{""PRIMARY"",null,""Abassin BADSHAH""}]",null,"[{""corp.disqual""}]","[{""31 Quernmore Close, Bromley, Kent, United Kingdom, BR1 4EL"",null,null,null,null,null,null}]","[{null,""1985-05-12""}]","[{null,""gb"",null}]","[{null,null,null,null,null,null,null,""OPEN-SANCTIONS"",""NK-25vyVFzt8vdJGgAXMRTwTJ""}]","[{""https://find-and-update.company-information.service.gov.uk/disqualified-officers/natural/mGquuTbmESWiRmHJPz1ObUwfDgk""}]","[{null,null,""Directorship"",""OPEN-SANCTIONS"",""NK-SKAADAiqiZ78JsJjeg72Te""}, {null,null,""Directorship"",""OPEN-SANCTIONS"",""NK-3p3mmVWmjwVtTfKchz4kNE""}]","""https://www.opensanctions.org/…",null
"""OPEN-SANCTIONS""","""sz:ds_open-sanctions_NK-3p3mmV…","""sz:Organization""","""2025-01-07T00:33:03""","[{""PRIMARY"",""LMAR (GB) LTD"",null}]",null,null,"[{""31 Quernmore Close, Bromley, Kent, United Kingdom, BR1 4EL"",null,null,null,null,null,""BUSINESS""}]",null,"[{""gb"",null,null}]","[{null,null,null,null,null,null,null,""OPEN-SANCTIONS"",""NK-3p3mmVWmjwVtTfKchz4kNE""}]",null,"[{""OPEN-SANCTIONS"",""NK-3p3mmVWmjwVtTfKchz4kNE"",null,null,null}]","""https://www.opensanctions.org/…",null
"""OPEN-SANCTIONS""","""sz:ds_open-sanctions_NK-auyPsL…","""sz:Organization""","""2024-03-03T19:51:29""","[{""PRIMARY"",""WANDLE HOLDINGS LIMITED"",null}]",null,"[{""sanction.linked""}]","[{""DEANA BEACH APTS, BLOCK A, Flat 212, Προμαχών Ελευθερίας, 33, 'Αγιος Αθανάσιος, 4103, Λεμεσός, Κύπρος"",null,null,null,null,null,""BUSINESS""}]","[{""2006-12-08"",null}]","[{""cy"",null,null}]","[{null,null,null,null,null,""C188266"",null,null,null}, {null,null,null,null,null,""HE188266"",null,null,null}, {null,null,null,null,null,null,null,""OPEN-SANCTIONS"",""NK-auyPsLrBzRoxjCRWgjBvas""}]",null,"[{""OPEN-SANCTIONS"",""NK-auyPsLrBzRoxjCRWgjBvas"",null,null,null}]","""https://opensanctions.org/enti…",null
"""OPEN-SANCTIONS""","""sz:ds_open-sanctions_NK-cf4Q3K…","""sz:Organization""","""2024-03-03T08:46:31""","[{""PRIMARY"",""POLYUS GOLD INTERNATIONAL LIMITED"",null}, {""ALIAS"",""KAZAKHGOLD GROUP LIMITED"",null}]",null,"[{""sanction.linked""}]","[{""3RD FLOOR CHARTER PLACE 23-27 SEATON PALCE, ST HELIER JE4 0WH"",null,null,null,null,null,""BUSINESS""}]","[{""2007-09-12"",null}]","[{""gb"",null,null}, {""je"",null,null}]","[{null,null,null,null,null,""FC027918"",null,null,null}, {null,null,null,null,null,null,null,""OPEN-SANCTIONS"",""NK-cf4Q3KcmUnQbt8Cy7iTtwK""}]","[{""http://business.data.gov.uk/id/company/FC027918""}]","[{""OPEN-SANCTIONS"",""NK-cf4Q3KcmUnQbt8Cy7iTtwK"",null,null,null}]","""https://opensanctions.org/enti…",null
"""OPEN-SANCTIONS""","""sz:ds_open-sanctions_NK-dNNN56…","""sz:Person""","""2025-02-10T15:38:02""","[{""PRIMARY"",null,""Firuza Nazimovna Kerimova""}, {""ALIAS"",null,""Firuza Nazimovna Khanbalaeva""}, … {""ALIAS"",null,""フィルザ・ケリモヴァ""}]","""F""","[{""role.rca""}, {""sanction""}]","[{""MOSCOW, RUS, 123430"",null,null,null,null,null,null}, {""Apt. 270, Build. 31, Pyatnitskoe Shosse, 123430 Moscow"",null,null,null,null,null,null}, … {""Apt 270, Build. 31, Pyatnitskoe Shosse, Moscow, 123430"",null,""Apt 270, Build. 31, Pyatnitskoe Shosse"",""Moscow"",""ru"",""123430"",null}]","[{null,""1967-12-22""}, {null,""1967-10-22""}]","[{null,""ru"",null}, {null,null,""ru""}]","[{""724348524"",null,null,null,null,null,null,null,null}, {null,null,null,null,null,""4512970434"",null,null,null}, … {null,null,null,null,null,null,null,""OPEN-SANCTIONS"",""NK-dNNN56A4ApVfUFvfzniLCF""}]","[{""https://sanctionssearch.ofac.treas.gov/Details.aspx?id=38277""}]","[{null,null,""Family"",""OPEN-SANCTIONS"",""Q447250""}]","""https://ww

In [4]:
# normalize the names from nested attributes
df: pl.DataFrame = df_src.explode("NAMES")

df = df.with_columns(
    pl.when(pl.col("NAMES").struct.field("NAME_TYPE") == "PRIMARY_NAME_FULL")
    .then(pl.col("NAMES").struct.field("NAME_TYPE"))
    .when(pl.col("NAMES").struct.field("NAME_TYPE") == "PRIMARY")
    .then(
        pl.coalesce(
            pl.col("NAMES").struct.field("NAME_FULL"),
            pl.col("NAMES").struct.field("NAME_ORG")
        )
    )
    .otherwise(None)
    .alias("descrip")
)

# select the columns to keep
df = df.select(
        pl.col("id"),
        pl.col("descrip"),
        pl.col("class"),
        pl.col("ADDRESSES").alias("addr"),
        pl.col("URL").alias("url"),
    ).drop_nulls(subset=["descrip"]).sort("id")

df = df.unique(subset = ["id", "descrip"]).sort("id")

# normalize the addresses from nested attributes
df = df.explode("addr")

df = df.with_columns(
    pl.when(pl.col("addr").is_not_null())
    .then(
        pl.coalesce(
            pl.col("addr").struct.field("ADDR_FULL"),
        )
    )
    .otherwise(None)
    .alias("addr")
)

df

id,descrip,class,addr,url
str,str,str,str,str
"""sz:ds_open-sanctions_NK-25vyVF…","""Abassin BADSHAH""","""sz:Person""","""31 Quernmore Close, Bromley, K…","""https://www.opensanctions.org/…"
"""sz:ds_open-sanctions_NK-3p3mmV…","""LMAR (GB) LTD""","""sz:Organization""","""31 Quernmore Close, Bromley, K…","""https://www.opensanctions.org/…"
"""sz:ds_open-sanctions_NK-L2UmsZ…","""Gulnara Suleimanova KERIMOVA""","""sz:Person""","""MOSCOW, RUS, 123430""","""https://www.opensanctions.org/…"
"""sz:ds_open-sanctions_NK-L2UmsZ…","""Gulnara Suleimanova KERIMOVA""","""sz:Person""","""Apt 270, Build. 31, Pyatnitsko…","""https://www.opensanctions.org/…"
"""sz:ds_open-sanctions_NK-L2UmsZ…","""Gulnara Suleimanova KERIMOVA""","""sz:Person""","""Apt 270, Build. 31, Pyatnitsko…","""https://www.opensanctions.org/…"
…,…,…,…,…
"""sz:ds_open-sanctions_rupep-com…","""Vencher Management Limited LLC""","""sz:Organization""","""ПЕР. СТАРОМОНЕТНЫЙ, Москва""","""https://opensanctions.org/enti…"
"""sz:ds_open-sanctions_rupep-com…","""Natsionalnaia Kinoset LLC""","""sz:Organization""","""переулок Старомонетны, Москва""","""https://opensanctions.org/enti…"
"""sz:ds_open-sanctions_rupep-com…","""Zareche-Estate LLC""","""sz:Organization""","""улица Народная, Москва""","""https://opensanctions.org/enti…"


Serialize the dataframe as the `os_data.csv` CSV file.

In [5]:
df.write_csv(pathlib.Path("os_data.csv"), separator = ",")

Also extract the risks from Open Sanctions, which will become a separate table.

In [6]:
df: pl.DataFrame = df_src.explode("RISKS")

df = df.with_columns(
    pl.when(pl.col("RISKS").is_not_null())
    .then(
        pl.coalesce(
            pl.col("RISKS").struct.field("TOPIC"),
        )
    )
    .otherwise(None)
    .alias("topic")
)

df = df.select(
    pl.col("id"),
    pl.col("topic"),
).drop_nulls(subset=["topic"])

df

id,topic
str,str
"""sz:ds_open-sanctions_NK-25vyVF…","""corp.disqual"""
"""sz:ds_open-sanctions_NK-auyPsL…","""sanction.linked"""
"""sz:ds_open-sanctions_NK-cf4Q3K…","""sanction.linked"""
"""sz:ds_open-sanctions_NK-dNNN56…","""role.rca"""
"""sz:ds_open-sanctions_NK-dNNN56…","""sanction"""
…,…
"""sz:ds_open-sanctions_ru-inn-77…","""sanction.linked"""
"""sz:ds_open-sanctions_rupep-com…","""sanction.linked"""
"""sz:ds_open-sanctions_rupep-com…","""sanction.linked"""


Serialize the dataframe as the `os_risk.csv` CSV file.

In [7]:
df.write_csv(pathlib.Path("os_risk.csv"), separator = ",")